# PandaSet cho bài lab 3D
Gắn PandaSet Dataset ở mục Input, sau đó chạy hai ô code theo thứ tự. Mặc định Notebook xuất tất cả cảnh có trong Input. Đặt SCENES = ["002"] để thử một cảnh.

Tải pandaset-lab.zip từ Output rồi giải nén vào gốc dự án. Mở viz/pandaset.html qua máy chủ của bài lab.


In [ ]:
"""Export PandaSet ego poses, camera indices and traffic cuboids for the browser lab."""

import argparse
import bisect
import json
import math
from pathlib import Path
import shutil

import pandas as pd


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def export_scene(folder, output, copy_camera=False):
    lidar = folder / "lidar"
    camera = folder / "camera" / "front_camera"
    poses = read_json(lidar / "poses.json")
    timestamps = read_json(lidar / "timestamps.json")
    camera_times = read_json(camera / "timestamps.json")
    origin, end = poses[0]["position"], poses[-1]["position"]
    yaw = math.atan2(end["y"] - origin["y"], end["x"] - origin["x"])
    forward_x, forward_y = math.cos(yaw), math.sin(yaw)

    def position(x, y):
        x, y = x - origin["x"], y - origin["y"]
        return [round(x * forward_x + y * forward_y, 3),
                round(1.75 + x * forward_y - y * forward_x, 3)]

    def heading(local_yaw):
        angle = yaw - local_yaw - math.pi / 2
        return round(math.atan2(math.sin(angle), math.cos(angle)), 5)

    frames, actors = [], {}
    if len(poses) != len(timestamps) or len(timestamps) < 2 or not camera_times:
        raise ValueError(f"Incomplete timestamps or poses in {folder}")
    if any(a >= b for a, b in zip(timestamps, timestamps[1:])) or any(
        a >= b for a, b in zip(camera_times, camera_times[1:])
    ):
        raise ValueError(f"Timestamps are not strictly increasing in {folder}")
    for index, (pose, timestamp) in enumerate(zip(poses, timestamps)):
        base = folder / "annotations" / "cuboids" / f"{index:02}.pkl"
        path = base if base.is_file() else base.with_suffix(".pkl.gz")
        # PandaSet distributes these as pandas pickles. Use a trusted dataset copy.
        cuboids = pd.read_pickle(path)
        cuboids = cuboids[cuboids["cuboids.sensor_id"] != 1]
        tracks = []
        for box in cuboids.to_dict(orient="records"):
            label = box["label"]
            if label in {"Pedestrian", "Pedestrian with Object", "Animals - Other"}:
                kind = "ped"
            elif label in {"Bicycle", "Motorcycle", "Personal Mobility Device"}:
                kind = "bike"
            elif any(word in label for word in ("Car", "Truck", "Bus", "Vehicle", "Trailer", "Tram", "Train")):
                kind = "car"
            else:
                continue
            x, z = position(box["position.x"], box["position.y"])
            if abs(z) > 32 or x < -40 or x > 170:
                continue
            key = box["uuid"]
            if key not in actors:
                actors[key] = [len(actors), label, kind, round(box["dimensions.y"], 3),
                               round(box["dimensions.x"], 3), round(box["dimensions.z"], 3)]
            tracks.append([actors[key][0], x, z, heading(box["yaw"])])
        orientation = pose["heading"]
        ego_yaw = math.atan2(2 * (orientation["w"] * orientation["z"] + orientation["x"] * orientation["y"]),
                             1 - 2 * (orientation["y"] ** 2 + orientation["z"] ** 2))
        frame_camera = bisect.bisect_left(camera_times, timestamp)
        frame_camera = min((max(0, frame_camera - 1), min(len(camera_times) - 1, frame_camera)),
                           key=lambda candidate: abs(camera_times[candidate] - timestamp))
        camera_name = f"{frame_camera:02}.jpg"
        if copy_camera:
            target = output.parent / folder.name / "camera" / "front_camera" / camera_name
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(camera / camera_name, target)
        frames.append({
            "t": round(timestamp - timestamps[0], 5),
            "ego": [*position(pose["position"]["x"], pose["position"]["y"]), heading(ego_yaw)],
            "camera": camera_name, "actors": tracks,
        })
    return {"id": folder.name, "actors": [actor[1:] for actor in actors.values()], "frames": frames}

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED, ZIP_STORED
from IPython.display import FileLink, display

# None = export every scene attached to Input. Use ["002"] for a quick test.
SCENES = None
inputs = Path("/kaggle/input")
folders = {file.parents[1].name: file.parents[1]
           for file in inputs.rglob("lidar/timestamps.json")
           if (file.parents[1] / "annotations" / "cuboids").is_dir()}
if not folders:
    raise RuntimeError("Add PandaSet Dataset to the Kaggle Notebook Input first.")

selected = sorted(folders) if SCENES is None else SCENES
missing = sorted(set(selected) - folders.keys())
if missing:
    raise ValueError(f"Scenes absent from Input: {missing}")

output = Path("/kaggle/working/pandaset-lab.zip")
with ZipFile(output, "w", compression=ZIP_DEFLATED) as package:
    results = []
    for scene_id in selected:
        folder = folders[scene_id]
        scene = export_scene(folder, output)
        results.append(scene)
        for image in sorted({frame["camera"] for frame in scene["frames"]}):
            relative = Path(scene_id) / "camera" / "front_camera" / image
            package.write(folder / "camera" / "front_camera" / image,
                          (Path("data/pandaset") / relative).as_posix(),
                          compress_type=ZIP_STORED)
        print(f"{scene_id}: {len(scene['frames'])} frames, {len(scene['actors'])} actors")
    package.writestr("data/pandaset/scenes.json",
                     json.dumps({"scenes": results}, ensure_ascii=False,
                                allow_nan=False, separators=(",", ":")))

print(f"Exported {len(results)} scenes to {output}")
display(FileLink(str(output.relative_to(Path.cwd())) if output.is_relative_to(Path.cwd()) else str(output)))
